# Feature Importance and SHAP lots

Machine learning models make decisions based on patterns in data. In this lab, you will explore how your trained model makes decisions by using feature importance and SHAP (SHapley Additive exPlanations), and examine which features your model relies on most, how they influence predictions, and whether that behavior is logical and fair.


You should already have:

- stedi_feature_pipeline.pkl
- stedi_best_model.pkl
- Transformed data files (e.g., X_train_transformed, X_test_transformed, y_test)

In [0]:
import joblib
import numpy as np
from pathlib import Path
from scipy.sparse import issparse

# Get this notebook’s Workspace path, then jump to repo root and into /artifacts
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_ws_path = ctx.notebookPath().get()                    # like: /Users/.../csai382.../notebooks/csai_lab_5_4
repo_ws_path = nb_ws_path.rsplit("/notebooks/", 1)[0]   # like: /Users/.../csai382...
ART = Path("/Workspace" + repo_ws_path) / "artifacts"

# Load artifacts (match your screenshot)
pipeline = joblib.load(ART / "stedi_feature_pipeline.pkl")

X_train_transformed = np.load(ART / "X_train_transformed.npy", allow_pickle=True)
X_test_transformed  = np.load(ART / "X_test_transformed.npy", allow_pickle=True)

y_train = joblib.load(ART / "y_train.pkl")
y_test  = joblib.load(ART / "y_test.pkl")

def to_float_matrix(arr: np.ndarray) -> np.ndarray:
   """
   Ensures that input arrays (possibly object-dtype, sparse, or 0-d) are converted to a 2-D float matrix.
   This is necessary because saved feature arrays may have inconsistent shapes or types after transformation,
   and ML models require numeric 2-D arrays for training and prediction.
   """
   if arr.ndim == 0:
       # Handle 0-d array directly
       arr = arr.item()
       if issparse(arr):
           arr = arr.toarray()
       arr = np.array(arr, dtype=float)
   elif arr.dtype == object:
       arr = np.array([
           x.toarray() if issparse(x) else np.array(x, dtype=float)
           for x in arr
       ])
       arr = np.vstack(arr)
   elif issparse(arr):
       arr = arr.toarray()
   else:
       arr = np.array(arr, dtype=float)
   return arr
X_train = to_float_matrix(X_train_transformed)
X_test = to_float_matrix(X_test_transformed)

y_train = np.ravel(y_train)
y_test = np.ravel(y_test)
model = joblib.load(ART / "stedi_best_model.pkl")
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [0]:
import numpy as np
import matplotlib.pyplot as plt

# grab the final estimator (last step in the pipeline)
#model = pipeline.steps[-1][1]

# LogisticRegression: coef_ shape is (n_classes, n_features) or (1, n_features)
coefs = model.coef_
if coefs.ndim == 2:
    # for multiclass: average absolute weight across classes
    importances = np.mean(np.abs(coefs), axis=0)
else:
    importances = np.abs(coefs)

importance_order = np.argsort(importances)[::-1]

# feature names from preprocessing (ColumnTransformer/OneHotEncoder, etc.)
try:
    feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
except Exception:
    feature_names = [f"feature_{i}" for i in range(importances.shape[0])]

# Step 2: print top features
top_k = 10
for idx in importance_order[:top_k]:
    print(feature_names[idx], ":", importances[idx])

# Step 3: bar chart (top features)
top_n = 15
top_idx = importance_order[:top_n][::-1]  # reverse so biggest is at top of chart
plt.figure(figsize=(10, 6))
plt.barh(range(top_n), importances[top_idx])
plt.yticks(range(top_n), np.array(feature_names)[top_idx])
plt.xlabel("Global importance (|coefficient|)")
plt.title("Top Feature Importances (Logistic Regression)")
plt.tight_layout()
plt.show()


First and foremost, the model values the distance traveled in Centimeters; if a specific entry has traveled distance, the model is more inclined to believe the data entry is in fact a movement rather than whitenoise. The other bars are simply which device it came from.

I am surprised it listed the devices as a deciding motivator, and I am not sure I would trust predictions made with this importance pattern alone, but I also cannot fathom how else the system could reliably tell what a non-whitenoise entry is.

In [0]:
%pip install shap


In [0]:
import shap
shap.initjs()

explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

shap.plots.beeswarm(shap_values, max_display=15)


Because we used a Logistic Regression model, we are not privy to anywhere near the level of features a Random Tree model would grant us. For example, Features are numbered without names. However, since we are plotting the same model we did before, then Feature 0 should also be num_distance_cm. As you can see, that feature alone moves the model an incredibly high amount.

In [0]:
i = 0  # pick a row
shap.initjs()
shap.plots.force(shap_values[i])


There's a little bit of math going on here, but that score (of 2.79, against the base value of 3.011) leaves about a 95% probability this specific entry is a step entry for the device which is named here as Feature 14, at least in the case of Index = 0.

While complicated, the values are used internally by the model to calculate the probability of any given entry being whitenoise or not.


# Reflection
Primarily, the distance changing is the single feature Logical Regression Testing is using to determine whether or not a specific entry is a step. I imagine a device wouldn't be a step if the device itself was not tracking movement, therefore it is a good tell of what is and isn't a step.

The SHAP force plot showed the Logistic Regression Model only used two values; the name of any given device, and that distance measurement. Both of those pushed the value down, if I understand the graph correctly, from 100% certainty to 95% certainty.

Whether or not this matches what a Human would expect is under debate; what would a human expect from any given data packet? As I look at the model, I am more certain that this is a good prediction on filtering whitenoise versus actual tracking data.

I definitely plan to include the feature importance chart and the SHAP summary plot in the Week 6 dashboard.